<a href="https://colab.research.google.com/github/adzetto/marine_analysis/blob/main/su-seviyesi/analiz_raporu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bozyazı mareograf istasyonu — deniz seviyesi analizi

**Konum:** 36.104° K, 32.948° D — Levantin (Doğu Akdeniz) kıyısı, Anamur yakını  
**Kaynak:** TUDES / Harita Genel Müdürlüğü, istasyon 11  
**Ölçüm:** 10 saniyede bir örneklenip 15 dakikada ortalanmış su seviyesi

Bu defter, üretilen sonuçları toplu olarak sunar. Ağır hesaplar
`05`–`08` betiklerinde yapılır ve diske yazılır; burada yalnızca okunup
gösterilir.

---

## İstenenler

| # | İstek | Nerede |
|---|---|---|
| 1 | Deniz seviyesindeki değişim | § 7 |
| 2 | Hatalı veriyi ayıkla, ortalaması MSL olsun | § 3, § 4 |
| 3 | Gelgit bileşenleri — genlik ve faz | § 5 |
| 4 | Gelgit düzeyleri — HAT … LAT | § 6 |
| 5 | Gelgit dışı su yükselmeleri — ortalama ve dağılım | § 7 |

In [ ]:
import os
if not os.path.exists('ortak.py'):
    !git clone -q https://github.com/adzetto/marine_analysis.git
    %cd marine_analysis/su-seviyesi
    !pip install -q -r requirements.txt

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown
from ortak import oku, kes, veri_yolu, PAPER_BAS, PAPER_BIT, MAKALE_GENLIK

pd.set_option('display.float_format', lambda v: f'{v:,.4f}')
print('hazir')

---
## 1. Kullanılan araçlar

| Kütüphane | Ne için |
|---|---|
| **UTide** (Codiga 2011) | Harmonik gelgit çözümü. MATLAB **T_TIDE**'ın (Pawlowicz ve ark., 2002) sürdürülen Python karşılığı; hocanın işaret ettiği program bu. Aynı en küçük kareler çözümü, aynı nodal düzeltmeler, aynı güven aralıkları. |
| `pandas` | Zaman serisi, yeniden örnekleme, hareketli istatistikler |
| `numpy` | Sayısal işlemler |
| `scipy.signal` | Yerel uç nokta bulma (yüksek/alçak su) |
| `requests` + `truststore` | TUDES portalından indirme |
| `matplotlib` + `scienceplots` | Figürler |
| `xlsxwriter` | Excel çıktısı |

Doğrulama kaynağı: **Öztürk & Yüksel (2023)**, *Regional Studies in
Marine Science* **61**, 102848 — aynı istasyon için 2009–2018 sonuçları
yayımlanmış.

---
## 2. Veri

Portalın `POST /Portal/VeriSorgula` ucu JSON döndürüyor; tek istekte en
fazla **60 gün** veriliyor, bu yüzden kayıt 55 günlük parçalar hâlinde
indirilip birleştiriliyor.

Kayıt **6 Temmuz 2009**'da başlıyor. (2005'ten itibaren ay ay tarandı;
öncesinde veri yok.)

In [ ]:
ham = oku('bozyazi_ham.dat')
temiz = oku('bozyazi_temiz.dat')
x = kes(temiz, PAPER_BAS, PAPER_BIT)

display(Markdown(f"""
| | |
|---|---|
| Ham ölçüm | {len(ham):,} |
| Kayıt aralığı | {ham.index.min():%Y-%m-%d} → {ham.index.max():%Y-%m-%d} |
| Ayıklama sonrası | {len(temiz):,} |
| **Analiz penceresi** | **{PAPER_BAS} → {PAPER_BIT}** ({len(x):,} ölçüm, {len(x)*15/60/24/365.25:.1f} yıl) |
"""))

y = temiz.groupby(temiz.index.year)
display(pd.DataFrame({'ortalama (m)': y.mean(), 'en düşük': y.min(),
                      'en yüksek': y.max(), 'ölçüm': y.count()}))

---
## 3. Hatalı verinin ayıklanması

Kayıtta iki farklı bozulma tipi var ve tek bir ölçüt ikisini birden
yakalamıyor:

* **Tekil sivri** — bir-iki ölçümlük ani sıçrama
* **Sürekli blok** — saatler/günler süren kaymış kayıt

Bu yüzden dört katman sırayla uygulanır. Sağlamlık için ortalama/standart
sapma yerine **medyan** ve **medyan mutlak sapma** kullanılır; tek bir uç
değer bunları kaydıramaz:

$$\hat{\sigma} = 1{,}4826 \cdot \operatorname{med}\big(|x_i - \operatorname{med}(x)|\big)$$

1,4826 katsayısı, normal dağılımda MAD'i standart sapmaya eşitler.

| Katman | Ölçüt | Hedef |
|---|---|---|
| 1 | $\lvert x - \operatorname{med}(x)\rvert > 2{,}0$ m | fiziksel olmayan okuma |
| 2 | 30 günlük medyandan $> 0{,}8$ m sapma | sürekli blok, datum kayması |
| 3 | 2 saatlik medyandan $> 0{,}10$ m sapma (iki geçiş) | tekil sivri |
| 4 | aynı değer $\geq 12$ kez üst üste | takılmış sensör |

1 günden kısa boşluklar zamana göre doğrusal interpolasyonla doldurulur —
makalenin izlediği kural.

**Ayıklama doğrulanmıştır:** hocanın gönderdiği Temmuz ve Eylül 2025
anomalileri tam olarak kalkıyor, sağlam on yılda silinen oran ise
%0,05–0,17 — yani gerçek sinyale dokunulmuyor.

In [ ]:
for f in ['figures/00_dogrulama_temmuz2025.png',
          'figures/00_dogrulama_eylul2025.png']:
    if os.path.exists(f):
        display(Image(f))

---
## 4. Ortalama deniz seviyesi (MSL)

Ayıklanmış serinin aritmetik ortalaması:

$$\mathrm{MSL} = \frac{1}{N}\sum_{i=1}^{N} \eta_i$$

Seviyeler istasyonun **yerel datumunda**; ülke yükseklik sistemine
bağlanmış değil. Gelgit düzeyleri ve artıklar göreli büyüklükler olduğu
için bu bir sorun yaratmaz.

In [ ]:
print(f'MSL (analiz penceresi) = {x.mean():.4f} m')
print(f'standart sapma         = {x.std():.4f} m')
print(f'ölçüm sayısı           = {len(x):,}')

---
## 5. Harmonik gelgit analizi

Gelgit, göreli konumları bilinen ay–yer–güneş hareketlerinden doğduğu
için **frekansları önceden bilinen** bir sinüsler toplamıdır. Bilinmeyen
yalnızca her bileşenin genliği ve fazıdır:

$$\eta(t) \;=\; a_0 \;+\; \sum_{k=1}^{K} f_k(t)\, A_k \cos\!\big[\omega_k t + (V_k + u_k)(t) - g_k\big]$$

| Simge | Anlam |
|---|---|
| $A_k,\; g_k$ | aranan genlik ve Greenwich fazı |
| $\omega_k$ | bileşenin açısal frekansı (astronomiden **bilinir**) |
| $V_k$ | denge gelgitinin faz argümanı |
| $f_k,\; u_k$ | **nodal düzeltmeler** — ayın yörünge düzleminin 18,61 yıllık salınımı genlikleri ±%4 kadar oynatır |

$\omega_k$ bilindiği için problem **doğrusaldır**. $x_k = A_k\cos g_k$ ve
$y_k = A_k \sin g_k$ konursa denklem bilinmeyenlerde doğrusal olur ve en
küçük karelerle çözülür:

$$\hat{\boldsymbol\beta} = \arg\min_{\boldsymbol\beta} \; \lVert \mathbf{y} - \mathbf{X}\boldsymbol\beta \rVert^2
\qquad\Longrightarrow\qquad
A_k = \sqrt{x_k^2 + y_k^2}, \quad g_k = \arctan\!\frac{y_k}{x_k}$$

Burada $\mathbf{X}$ 300.000 satırlı tasarım matrisidir — çözümün birkaç
GB bellek istemesinin sebebi budur.

### Hangi bileşenler anlamlı

Makaledeki ölçüt: sinyal–gürültü oranı

$$\mathrm{SNR}_k = \left(\frac{A_k}{\sigma_{A_k}}\right)^{\!2}, \qquad \mathrm{SNR} > 2$$

### Gelgitin tipi

**Form faktörü** (Pugh 1987) — günlük bileşenlerin yarı-günlüklere oranı:

$$F = \frac{H_{K_1} + H_{O_1}}{H_{M_2} + H_{S_2}}$$

$F < 0{,}25$ yarı-günlük · $0{,}25\!-\!1{,}5$ karışık ağırlıklı yarı-günlük ·
$1{,}5\!-\!3$ karışık ağırlıklı günlük · $>3$ günlük

**Enerji faktörü** (Medvedev ve ark. 2016) — bütün bileşenleri hesaba katar:

$$E = \frac{\sum_j H^2_{D_j}}{\sum_j H^2_{SD_j}}$$

In [ ]:
bil = pd.read_csv('tables/01_gelgit_bilesenleri_makale_penceresi.csv')
bil['makale_cm'] = bil.bilesen.map(MAKALE_GENLIK)
bil['genlik_m'] = bil.genlik_cm / 100

print('En büyük 14 bileşen:')
display(bil.head(14)[['bilesen', 'genlik_m', 'genlik_cm',
                      'faz_derece', 'SNR', 'makale_cm']])

k = bil.dropna(subset=['makale_cm']).copy()
k['fark_cm'] = k.genlik_cm - k.makale_cm
k['fark_%'] = 100 * k.fark_cm / k.makale_cm
print(f'\nMakaleyle karşılaştırma — ortalama mutlak fark: '
      f'{k.fark_cm.abs().mean():.3f} cm')
display(k[['bilesen', 'makale_cm', 'genlik_cm', 'fark_cm', 'fark_%']])

In [ ]:
g = dict(zip(bil.bilesen, bil.genlik_cm))
F = (g.get('K1',0) + g.get('O1',0)) / (g.get('M2',0) + g.get('S2',0))
D  = ['K1','O1','P1','S1']
SD = ['M2','S2','N2','K2']
E = sum(g.get(c,0)**2 for c in D) / sum(g.get(c,0)**2 for c in SD)
print(f'Form faktörü   F = {F:.3f}   (makale 0.30)')
print(f'Enerji fakt.   E = {E:.3f}   (makale 0.09)')
print('Tip: karışık, ağırlıklı yarı-günlük (MSD)')

ana = ['M2','S2','N2','K2','K1','O1','P1','S1','SSA','SA']
v = [g.get(c, 0) for c in ana]
m = [MAKALE_GENLIK[c] for c in ana]
i = np.arange(len(ana)); w = 0.4
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(i - w/2, m, w, label='makale (2009-2018)', color='#8c8c8c')
ax.bar(i + w/2, v, w, label='bu çalışma', color='#1f4e9c')
ax.set_xticks(i); ax.set_xticklabels(ana)
ax.set_ylabel('genlik (cm)'); ax.legend(); ax.grid(alpha=.3, axis='y')
ax.set_title('Gelgit bileşenleri — yayımlanmış değerlerle karşılaştırma')
plt.tight_layout(); plt.show()

---
## 6. Gelgit düzeyleri

Düzeyler doğrudan ölçümden değil, çözülen bileşenlerden **üretilen**
gelgit öngörüsünden çıkarılır. Sebebi HAT ve LAT'ın tanımı gereği
*astronomik* uç değerler olması: fırtına kabarması gibi meteorolojik
katkılar girmemelidir.

Öngörü **19 yıl** boyunca üretilir; nodal döngü 18,61 yıl olduğu için daha
kısa bir pencere HAT/LAT'ı sistematik olarak küçük verir.

Bahar (*spring*) ve ölüm (*neap*) gelgiti ayrımı veriye dayalı yapılır:
$M_2$ ile $S_2$ arasındaki vurum

$$T_{\text{vurum}} = \frac{2\pi}{\omega_{M_2} - \omega_{S_2}} \approx 14{,}77 \text{ gün}$$

Günlük gelgit genliğinin yerel tepe noktaları bahar, yerel dip noktaları
ölüm gelgitidir. Klasik yaklaşıklıklar da karşılaştırma için verilir:

$$\mathrm{MHWS} \approx \mathrm{MSL} + (H_{M_2} + H_{S_2}), \qquad
\mathrm{MHWN} \approx \mathrm{MSL} + (H_{M_2} - H_{S_2})$$

In [ ]:
duz = pd.read_csv('tables/02_gelgit_duzeyleri.csv')
display(duz)

d = duz.set_index('duzey').su_seviyesi_m_MSL
print(f"Ortalama bahar gelgiti aralığı (MHWS-MLWS) = "
      f"{(d['MHWS']-d['MLWS'])*100:.1f} cm")
print(f"Azami astronomik aralık (HAT-LAT)          = "
      f"{(d['HAT']-d['LAT'])*100:.1f} cm")

fig, ax = plt.subplots(figsize=(7, 4.6))
sira = ['HAT','MHWS','MHHW','MHW','MHWN','MSL','MLWN','MLW','MLLW',
        'MLWS','LAT']
vals = [d[s] for s in sira]
renk = ['#c0392b' if v > 0 else ('#1f4e9c' if v < 0 else '#333')
        for v in vals]
ax.barh(range(len(sira)), vals, color=renk)
ax.set_yticks(range(len(sira))); ax.set_yticklabels(sira)
ax.invert_yaxis(); ax.axvline(0, color='k', lw=.8)
ax.set_xlabel('MSL\'e göre su seviyesi (m)')
ax.set_title('Bozyazı — gelgit düzeyleri')
ax.grid(alpha=.3, axis='x')
for i, v in enumerate(vals):
    ax.text(v + (0.006 if v >= 0 else -0.006), i, f'{v:+.3f}',
            va='center', ha='left' if v >= 0 else 'right', fontsize=8)
plt.tight_layout(); plt.show()

### SA / SSA sorunu

Bu istasyonda yıllık bileşen $S_A \approx 9{,}4$ cm ve yarı-yıllık
$S_{SA} \approx 4{,}2$ cm — yani **$M_2$'den büyükler**. Ancak bunlar
büyük ölçüde *ışınımsal*: suyun mevsimlik ısınıp genleşmesi, gravitasyonel
gelgit değil.

Denizcilik gelgit tabloları (hocanın örnek verdiği Famagusta IHO tablosu
gibi) kısa periyotlu gravitasyonel gelgiti verir. İki sürüm de
hesaplanmıştır; hangisinin raporlanacağı bir tercih meselesidir.

---
## 7. Gelgit dışı (non-tidal) su seviyesi

Ölçülenden öngörülen gelgit çıkarılınca geriye meteorolojik ve hidrolojik
katkı kalır — hocanın "gelgitin dışında fırtına ile yükselen su seviyesi"
dediği büyüklük:

$$\eta_{nt}(t) \;=\; \eta_{\text{ölçülen}}(t) \;-\; \eta_{\text{gelgit}}(t)$$

### Ortalama neden işe yaramıyor

Hoca "129 cm gibi bir kez olmuş max değil, ortalama lazım" dedi. Ancak
$\eta_{nt}$'nin ortalaması **tanımı gereği sıfırdır**: seriden hem gelgit
hem de ortalama seviye çıkarılmıştır.

$$\overline{\eta_{nt}} \equiv 0$$

Nitekim makalenin kendi Tablo 3'ünde "Mean" sütunu **18 istasyonun hepsi
için** tek bir değer olarak $\approx 0$ yazılmıştır. Yani düz ortalama
tasarım için bilgi taşımaz.

Dağılımı temsil eden büyüklük **standart sapmadır** (makale Bozyazı için
10,43 cm) ve **aşılma yüzdelikleridir**. İkisi de aşağıda verilmiştir.

### Gelgit baskınlık oranı

Makale Denklem (3) — gelgit salınımının enerjisinin gelgit dışına oranı:

$$TD = \sqrt{\frac{\sum_j H^2_{\text{gelgit},j}}{\sum_j H^2_{\text{gelgit dışı},j}}}
= \frac{\mathrm{RMS}(\eta_{\text{gelgit}})}{\mathrm{RMS}(\eta_{nt})}$$

$TD > 1$ gelgitin baskın olduğu anlamına gelir.

In [ ]:
oz = pd.read_csv('tables/03_nontidal_ozet.csv')
display(oz)
yuz = pd.read_csv('tables/04_nontidal_yuzdelikler_makale_penceresi.csv')
yuz['seviye_m'] = yuz.seviye_cm / 100
display(yuz)

a = oz.iloc[0]
display(Markdown(f"""
**Makale Tablo 3 ile karşılaştırma (Bozyazı):**

| Büyüklük | Makale | Bu çalışma |
|---|---|---|
| Ortalama | ≈ 0 | {a.ortalama_cm:+.4f} cm |
| Standart sapma | 10,43 cm | {a.std_cm:.2f} cm |
| $TD$ | 1,06 | {a.TD:.2f} |
| Azami aralık | 129 cm | {a.aralik_cm:.1f} cm |
"""))

In [ ]:
for f, a in [('figures/01_nontidal_pdf_cdf_makale_penceresi.png',
              'Makaledeki gibi 10 kutu — Fig. 6 ile doğrudan karşılaştırılabilir. '
              'Sol üstteki küçük harita istasyonu yıldızla gösteriyor (M_MAP stili).'),
             ('figures/01b_nontidal_pdf_cdf_ince_makale_penceresi.png',
              'Aynı dağılım, 2 cm kutulama — dağılımın şeklini daha iyi gösterir'),
             ('figures/02_nontidal_zaman_serisi_makale_penceresi.png',
              'Zaman serisi (makale Fig. 5 düzeni)')]:
    if os.path.exists(f):
        display(Markdown(f'**{a}**'))
        display(Image(f))

### Grafiğin verisi

Yukarıdaki eğrilerin ta kendisi olan sayılar CSV'ye ve Excel kitabına da
yazılır — kitapta bu hücrelerden kurulu **canlı bir Excel grafiği** vardır,
hücreler değiştirilirse grafik de değişir.

Sütunlar: kutu sınırları, kutu ortası, ham sayım, PDF ve CDF. Burada

$$\mathrm{PDF}_i = \frac{n_i}{\sum_j n_j}, \qquad
\mathrm{CDF}_i = \sum_{j \leq i} \mathrm{PDF}_j$$

yani PDF olasılık *yoğunluğu* değil, kutu başına **olasılık**tır — makale
Fig. 6 ile aynı normalizasyon (MATLAB `hist` + `'probability'`). Bu yüzden
kutu genişliği değişince tepe değeri de değişir; iki sürümün tepe
değerlerinin farklı olmasının sebebi budur.

In [ ]:
h = pd.read_csv('tables/01_nontidal_pdf_cdf_makale_penceresi_veri.csv')
print(f'{len(h)} kutu, genislik {(h.kutu_ust_m[0]-h.kutu_alt_m[0])*100:.1f} cm')
display(h)
print(f'PDF toplami = {h.PDF.sum():.6f}   (1 olmali)')
print(f'CDF sonu    = {h.CDF.iloc[-1]:.6f}   (1 olmali)')

---
## 8. Deniz seviyesindeki değişim

Yıllık ortalamalar **doğrusal değil, V biçimli**: 2017'de minimum, iki
yanda yükseliş. Bir V'nin inen koluna doğru çekilen bir doğru eğilim
değil, o kolun eğimidir — nitekim seçilen pencere işaretin yönünü
belirliyor:

$$\text{2010–2019: } -11{,}3\ \mathrm{mm/yıl} \qquad
\text{2021–2026: } +4{,}6\ \mathrm{mm/yıl}$$

Bu V'nin gerçek mi yoksa cihaz kaynaklı mı olduğunu ayırt etmek için aynı
kıyıdaki üç komşu istasyon (Taşucu, Erdemli, Antalya) indirilip aynı
ayıklamadan geçirildi. **2017 minimumu dördünde de var** ($r = 0{,}75$–$0{,}81$),
yani işaret **bölgesel ve gerçek**.

Buna karşılık 16 yıl, on yıllık değişkenliğin baskın olduğu bir seride
trend kestirmek için kısadır; güvenilir bir deniz seviyesi trendi genelde
30+ yıl ister. Bağımsız doğrulama yolu: TUDES aylık ortalamalarını
**PSMSL**'e gönderiyor ve PSMSL kayıtları röpere indirgenmiş (RLR).

In [ ]:
if os.path.exists('figures/03_istasyon_karsilastirma.png'):
    display(Image('figures/03_istasyon_karsilastirma.png'))
if os.path.exists('tables/06_istasyon_yillik_sapma.csv'):
    display(pd.read_csv('tables/06_istasyon_yillik_sapma.csv'))

---
## 9. Excel çıktısı

Bütün seriler, tablolar ve figürler tek bir kitapta toplanır.

In [ ]:
!python -u 10_excel_olustur.py

In [ ]:
!zip -qr /content/bozyazi_sonuclar.zip tables figures data *.xlsx 2>/dev/null || zip -qr bozyazi_sonuclar.zip tables figures data *.xlsx
try:
    from google.colab import files
    files.download('/content/bozyazi_sonuclar.zip')
except Exception as e:
    print('Colab disi ortam:', e)